# imports

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import itertools

In [ ]:
import sys
sys.path.append("code")
import final_functions as ff

In [ ]:
import importlib
importlib.reload(ff)
pass

# saving results

In [ ]:
simulation_results_file_path = os.path.join(ff.get_file_path("project_root"), "final", "results", "simulations")

p1_values = [1, 0.8, 0.5]
p2_values = [1, 0.8, 0.5]
num_instances = 5

results = [] # gonna be list of lists

combos = list(itertools.product(p1_values, p2_values, range(1, num_instances + 1)))
for p1, p2, seed in tqdm(combos):
    results.append(ff.run_simulation_and_save_result(p1, p2, seed, os.path.join(simulation_results_file_path, f"{p1}_{p2}", f"{seed}")))

df = pd.DataFrame(results, columns=["p1", "p2", "seed", "our_ilp_max_2_alt_alteration_recall", "our_ilp_max_2_alt_alteration_precision", "our_ilp_max_2_alt_pathway_accuracy", "our_ilp_max_2_alt_dependency_accuracy", "our_ilp_no_max_alt_alteration_recall", "our_ilp_no_max_alt_alteration_precision", "our_ilp_no_max_alt_pathway_accuracy", "our_ilp_no_max_alt_dependency_accuracy"])

df.to_csv(os.path.join(simulation_results_file_path, "summary.csv"), index=False)

# testing

## pathways = [['IDH1', 'IDH2'], ['SRSF2'], ['NRAS', 'KRAS']], AML topologies, 1, 1, n/a, n/a

In [ ]:
# simulate

topologies = [trees[0] for trees in ff.get_patient_trees_lists("AML")]
alterations = set(ff.get_all_alterations("AML")) - {" "}

pathways = [['IDH1', 'IDH2'], ['SRSF2'], ['NRAS', 'KRAS']]

simulated_trees = ff.simulate_tumor_trees(seed=1, num_trees=len(topologies), pathways=pathways, alterations=alterations, tree_follows_PM_prob=1, node_follows_PM_prob=1, topology_type="given", topologies=topologies, branch_prob=None, progression_prob=None)

In [ ]:
# showing simulated trees

ff.show_patient_trees([[tree] for tree in simulated_trees])

In [ ]:
# see how many strictly and partially consistent with the CBNPM, and with each trajectory

print(f"Original linear progression model with pathways: {pathways}")
print()
ff.show_strictly_and_partially_consistent_patients(ff.make_CBNPM({i: pathway for i, pathway in enumerate(pathways, start=1)}, [(k, k+1) for k in range(1, len(pathways))]), [[tree] for tree in simulated_trees])
print()

trajectories = ff.get_trajectories(pathways)

for i, trajectory in enumerate(trajectories, start=1):
    print(f"Trajectory {i}: ", end="")
    ff.print_trajectory(trajectory)
    print()
    ff.show_strictly_and_partially_consistent_patients(trajectory, [[tree] for tree in simulated_trees])
    print()

In [ ]:
# run our ilp to get pathway progression model

patient_trees = [[tree] for tree in simulated_trees]
alterations = alterations
num_levels = len(pathways)
num_pathways = len(pathways)
num_dependencies = len(pathways) - 1
max_solutions = 1
max_alterations_per_pathway = None

ilp_results_simulated_1_pm = ff.problem_1_1(patient_trees, alterations, num_pathways, num_dependencies, num_levels, max_solutions=max_solutions, max_alterations_per_pathway=max_alterations_per_pathway)

In [ ]:
ff.problem_1_1_display_results(ilp_results_simulated_1_pm)

In [ ]:
# run ilp to recover trajectories

patient_trees = [[tree] for tree in simulated_trees]
alterations = alterations
num_levels = len(pathways)
num_pathways = len(pathways)
num_dependencies = len(pathways) - 1
max_solutions = 1
max_alterations_per_pathway = 1

ilp_results_simulated_1_trajectory = ff.problem_1_1(patient_trees, alterations, num_pathways, num_dependencies, num_levels, max_solutions=max_solutions, max_alterations_per_pathway=max_alterations_per_pathway)

In [ ]:
ff.problem_1_1_display_results(ilp_results_simulated_1_trajectory)

In [ ]:
MASTRO_format_trees_file_path = os.path.join(ff.get_file_path("MASTRO_format_trees_files"), "simulation_1.txt")
MASTRO_results_simulated_1 = ff.run_MASTRO(simulated_trees, MASTRO_format_trees_file_path)

In [ ]:
ff.show_MASTRO_results(MASTRO_results_simulated_1)